#### <u>**Bài toán:**</u> Sử dụng dữ liệu về khách hàng của các ngân hàng Châu Âu để dự đoán xem khách hàng có trung thành với ngân hàng hay không, có tiếp tục sử dụng dịch vụ của ngân hàng hay không, hay họ sẽ đóng tài khoản và rời đi ngân hàng khác.

In [1]:
import pandas as pd

# Read and prepare the dataset
df = pd.read_csv(
    "../../../../datasets/Customer-Churn-Records.csv",
    header=0,
    na_values="NA",
    comment="\t",
    sep=",",
    skipinitialspace=True,
    encoding='utf-8'    # Added encoding to handle special characters
)

# Drop unnecessary columns to avoid noise in the model
df.drop(
    columns=[
        "RowNumber",
        "CustomerId",
        "Surname",
        "Complain",
        "Satisfaction Score",
        "Card Type",
        "Point Earned",
    ],
    inplace=True,   # Added inplace=True to modify df directly
)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  object 
 2   Gender           10000 non-null  object 
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


In [2]:
from sklearn.preprocessing import LabelEncoder

ecd = LabelEncoder()
df['Gender'] = ecd.fit_transform(df['Gender'].values)
df = pd.get_dummies(df)

In [3]:
X = df.drop("Exited", axis=1)
y = df["Exited"]

In [4]:
# from all modules and functions
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [5]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#### Sử dụng Stacking cho bài toán phân loại

In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Define base models
base_models = [
    ("KNN", KNeighborsClassifier(n_neighbors=3)),
    ("Decision Tree", DecisionTreeClassifier(max_depth=2, random_state=42)),
]

In [ ]:
from sklearn.linear_model import LogisticRegression

# Define meta model (Logistic Regression in this example)
# max_iter increased to ensure convergence
# random_state added for reproducibility
meta_model = LogisticRegression(max_iter=500, random_state=42)
# meta_model = DecisionTreeClassifier(max_depth=2, random_state=42)

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score

# Create the StackingClassifier
stacking_clf = StackingClassifier(
    estimators=base_models,  # List of base models
    final_estimator=meta_model,  # Meta model
    cv=5,  # Number of cross-validation folds
)

# Fit the stacking model on the training data
stacking_clf.fit(X_train, y_train)

# Make predictions on the test data
y_pred_stack = stacking_clf.predict(X_test)

# Calculate accuracy
accuracy_stack = accuracy_score(y_test, y_pred_stack)
print(f"Stacking Classifier Accuracy: {accuracy_stack:.4f}")

Stacking Classifier Accuracy: 0.5100


Nếu sử dụng riêng lẻ một mình mô hình KNN hoặc Decision Tree với cùng 1 hyperparameter thì kết quả dự đoán như thế nào

In [9]:
# Evaluate individual base models for comparison
for name, model in base_models:
    model.fit(X_train, y_train)
    y_pred_base = model.predict(X_test)
    accuracy_base = accuracy_score(y_test, y_pred_base)
    print(f"{name} Accuracy: {accuracy_base:.4f}")

KNN Accuracy: 0.4825
Decision Tree Accuracy: 0.5095


Sử dụng Random Forest và Gradient Boosting làm base learner

In [10]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Define other ensemble models for comparison
base_models = [
    ("Random Forest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("Gradient Boosting", GradientBoostingClassifier(n_estimators=100, random_state=42)),
]

# Define meta model (Logistic Regression in this example)
meta_model = LogisticRegression(max_iter=500, random_state=42)

# Create the StackingClassifier
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5
)

# Fit the stacking model on the training data
stacking_clf.fit(X_train, y_train)

# Make predictions on the test data
y_pred_stack = stacking_clf.predict(X_test)

# Calculate accuracy
accuracy_stack = accuracy_score(y_test, y_pred_stack)
print(f"Stacking Classifier Accuracy (RF + GB): {accuracy_stack:.4f}")

Stacking Classifier Accuracy (RF + GB): 0.5040


In [11]:
for name, model in base_models:
    model.fit(X_train, y_train)
    y_pred_base = model.predict(X_test)
    accuracy_base = accuracy_score(y_test, y_pred_base)
    print(f"{name} Accuracy: {accuracy_base:.4f}")

Random Forest Accuracy: 0.4935
Gradient Boosting Accuracy: 0.4920
